In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

/root/miniconda3/envs/transformers/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1、下载数据集

In [2]:
from datasets import load_dataset
ds = load_dataset("llamafactory/alpaca_zh", cache_dir="../data",split = "train")

In [3]:
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 51155
})

In [4]:
ds = ds.train_test_split(test_size=0.8,seed = 42)
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 40924
    })
})

In [5]:
ds['train'][5]

{'instruction': '解释气候变化对环境的两个影响',
 'input': '',
 'output': '气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。'}

## 2、数据预处理

In [6]:
#模型下载
from modelscope import snapshot_download
model_dir = snapshot_download('qwen/Qwen2-0.5B', cache_dir="./models")

In [7]:
model_path = "./models/qwen/Qwen2-0___5B"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer

Qwen2TokenizerFast(name_or_path='./models/qwen/Qwen2-0___5B', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [9]:
def process_func(datasets, tokenizer, max_length=256):
    """
    处理输入示例，合并用户的指令和输入字段，然后对输入和输出进行分词。
    该函数准备好训练自回归语言模型的数据格式。

    参数：
    - datasets (dict): 包含 'instruction'、'input' 和 'output' 字段的字典。
    - tokenizer (AutoTokenizer): 用于分词的 tokenizer。
    - max_length (int): 序列的最大长度。

    返回：
    - dict: 准备好的训练数据，包括 tokenized 的输入和标签。
    """
    # 1. 生成输入部分，包括 "Human: " 和 "Assistant: " 标签
    if datasets['input'].strip() == "":
        combined_input = "用户: " + datasets['instruction'] + "\n\n 机器人: "
    else:
        combined_input = "用户: " + datasets['instruction'] + "\n" + datasets['input'] + "\n\n 机器人: "

    # 2. 生成输出部分，并在最后加上终止标记
    output_text = datasets["output"] + tokenizer.eos_token  # 加上 eos_token 标记回复的结束

    # 3. 合并输入和输出作为模型的完整输入序列
    full_input = combined_input + output_text

    # 4. 对完整输入进行分词处理，确保生成 input_ids 和 attention_mask
    encodings = tokenizer(
        full_input, 
        max_length=max_length, 
        truncation=True, 
        padding="max_length", 
        return_tensors='pt'
    )

    # 5. 复制 input_ids 来生成标签
    labels = encodings["input_ids"].clone()

    # 6. 计算用户输入部分的长度（即需要忽略的部分）
    user_input_len = len(tokenizer(combined_input, truncation=True, max_length=max_length)["input_ids"])

    # 7. 将用户输入部分的标签设置为 -100，忽略这部分的损失
    labels[:, :user_input_len] = -100

    # 8. 返回 input_ids、attention_mask 和 labels，用于训练模型
    return {
        'input_ids': encodings['input_ids'].squeeze(0),  # 输入序列
        'attention_mask': encodings['attention_mask'].squeeze(0),  # 注意力掩码
        'labels': labels.squeeze(0)  # 标签，忽略用户输入部分
    }


In [10]:
# 假设 ds 是你的数据集，并且 tokenizer 已经定义
tokenized_ds = ds.map(
    process_func,
    remove_columns=['instruction', 'input', 'output'],  # 删除不需要的原始列
    fn_kwargs={'tokenizer': tokenizer},  # 将 tokenizer 传递给 process_func
)

tokenized_ds

Map:   0%|          | 0/10231 [00:00<?, ? examples/s]

Map: 100%|██████████| 40924/40924 [00:43<00:00, 943.86 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40924
    })
})

In [11]:
tokenizer.decode(tokenized_ds["train"][5]["input_ids"])

'用户: 解释气候变化对环境的两个影响\n\n 机器人: 气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><

In [12]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds["train"][5]["labels"])))

'��候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|

In [42]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

dl = DataLoader(tokenized_ds['train'], batch_size=1, collate_fn=DataCollatorForLanguageModeling(tokenizer, mlm=False))

In [43]:
next(enumerate(dl))

(0,
 {'input_ids': tensor([[ 20002,     25,  93685,  83744,  46944, 100006, 101987, 102314,  49238,
          103310,  47764, 100032, 111564,   3407,    220, 104354,     25,    220,
          102314,  18830,     17,     15,     21,  99922,  67071, 107914,  33108,
           64272, 100049,  64064,   9370, 110013,   1773,  77557,   3837, 103335,
          107914,  20412,  63703,  99408, 110013,   3837,  99195,  38989, 101843,
           99278,  33108,  77540,  34718, 115251,  87531, 107725, 106888,  64064,
            3837,  73670, 101884, 109838, 101079,   3837,  29524, 102683, 100036,
           33108, 100867,  76313,   1773,  80443, 100001,  64064,   3837, 103335,
          107914,  44063, 101068,  99553, 101099, 109988, 118280,   1773, 151643,
          151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
          151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
          151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643

In [44]:
print(next(enumerate(dl))[1]['labels'][0])

tensor([ 20002,     25,  93685,  83744,  46944, 100006, 101987, 102314,  49238,
        103310,  47764, 100032, 111564,   3407,    220, 104354,     25,    220,
        102314,  18830,     17,     15,     21,  99922,  67071, 107914,  33108,
         64272, 100049,  64064,   9370, 110013,   1773,  77557,   3837, 103335,
        107914,  20412,  63703,  99408, 110013,   3837,  99195,  38989, 101843,
         99278,  33108,  77540,  34718, 115251,  87531, 107725, 106888,  64064,
          3837,  73670, 101884, 109838, 101079,   3837,  29524, 102683, 100036,
         33108, 100867,  76313,   1773,  80443, 100001,  64064,   3837, 103335,
        107914,  44063, 101068,  99553, 101099, 109988, 118280,   1773,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100, 

In [45]:
tokenizer.eos_token, tokenizer.eos_token_id

('<|endoftext|>', 151643)

In [47]:
# 获取第一个样本的 labels
labels = next(enumerate(dl))[1]['labels'][0]

# 去掉 -100 的值（这是忽略损失的 token）
valid_labels = labels[labels != -100]

# 对有效的 token 进行解码
decoded_text = tokenizer.decode(valid_labels, skip_special_tokens=True)

print(decoded_text)


用户: 提供一个能够展示人体解剖学知识的例子。

 机器人: 人体有206块由关节和软骨连接的骨头。例如，膝关节是四根骨头，几条韧带和两片半月板的高度复杂的连接，可以实现广泛的运动，如屈曲和伸展。没有这些连接，膝关节将无法提供身体所需的灵活性。


## 3、创建模型

In [17]:
model = AutoModelForCausalLM.from_pretrained(model_path)

In [38]:
model01 = AutoModelForCausalLM.from_pretrained(model_path)

In [39]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model01, tokenizer=tokenizer, device=0)

In [40]:
ipt = "用户: {}\n{}".format("解释气候变化对环境的两个影响\?", "").strip() + "\n\n机器人: "

In [41]:
re = pipe(ipt, max_length=256, do_sample=True, )[0]["generated_text"]
print(re)

Both `max_new_tokens` (=2048) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


用户: 解释气候变化对环境的两个影响\?

机器人: 1)大气中温室气体含量增加导致地球温度上升 2)森林和河流大量消失,导致洪水、干旱发生


## 4、创建训练参数

In [21]:
args = TrainingArguments(
    output_dir="./models_for_chatbot",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # 每4步累计一次
    per_device_eval_batch_size=32,
    logging_steps=20,
    num_train_epochs=1,
    report_to=['tensorboard'],
)

In [22]:
#import torch
#torch.cuda.empty_cache()

## 5、创建训练器并训练

In [23]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"], 
    eval_dataset=tokenized_ds["test"],     
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [24]:
trainer.train()

Step,Training Loss
20,1.515500
40,0.515500
60,0.565200
80,0.516600
100,0.533200
120,0.540500
140,0.495800
160,0.520000
180,0.522800
200,0.494200


TrainOutput(global_step=319, training_loss=0.5789685652921193, metrics={'train_runtime': 1082.6113, 'train_samples_per_second': 9.45, 'train_steps_per_second': 0.295, 'total_flos': 5611659152326656.0, 'train_loss': 0.5789685652921193, 'epoch': 0.9976544175136826})

## 6、预测

In [33]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [26]:
#ipt = ": {}\n{}".format("简述文章的主要观点：「人工智能如何帮助应对气候变化」", "").strip() + "\n\nAssistant: "

In [34]:
ipt = "用户: {}\n{}".format("解释气候变化对环境的两个影响", "").strip() + "\n\n机器人: "

In [35]:
re = pipe(ipt, max_length=256, do_sample=True, )[0]["generated_text"]

Both `max_new_tokens` (=2048) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [36]:
print(re)

用户: 解释气候变化对环境的两个影响

机器人: 1. 温度升高影响生物多样性。随着温度升高，海平面上升，海洋动物的栖息地正在减少。
2. 可能有海平面上升，导致沿海地区被洪水淹没，或极端天气，导致农作物或树木受到破坏的损失。


In [37]:
print(ds['train'][5]["output"])

气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。
